# Linear SVM grid search

Nipun Yonjan

I am tuning a linear SVM on the shared split. The prep pipeline stays inside the grid search, so the scaler gets refit on each training fold. I score the winner on dev. Test is for the conclusion notebook.


## 1. Load the split

prepare_datasets() drops the duplicate URLs and gives us the same train, dev, and test split. I send the raw training table into the search. Scaling happens inside the pipeline, not before it.

The grid is four values of C and two class-weight settings. That covers both SVMs from the investigations: C=1.0 with no class weight, and my C=0.01 with balanced weights. The thing being optimized is phishing F1.


In [1]:
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from dataset_preparation import build_preparation_pipeline, prepare_datasets

data = prepare_datasets()

print("Train rows:", len(data["y_train"]))
print("Dev rows:", len(data["y_dev"]))
print("Train phishing rate: {:.2%}".format(data["y_train"].mean()))
print("Dev phishing rate:   {:.2%}".format(data["y_dev"].mean()))
print("Columns going into the search:", data["X_train"].shape[1])

model = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual=False,
    max_iter=5000,
    random_state=42,
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

full_pipeline = Pipeline(
    steps=[
        ("prepare", build_preparation_pipeline()),
        ("model", model),
    ]
)

search = GridSearchCV(
    full_pipeline,
    param_grid,
    scoring="f1",
    cv=3,
)

print("Grid combinations:", len(ParameterGrid(param_grid)))
print("Search has been fit:", hasattr(search, "best_params_"))

Train rows: 69138
Dev rows: 23046
Train phishing rate: 14.40%
Dev phishing rate:   14.40%
Columns going into the search: 25
Grid combinations: 8
Search has been fit: False


Train and dev are both about 14.40% phishing. That is 8 settings and 3-fold CV. I have not fit the search yet, and I have not used dev or test.


## 2. Fit the search

Each of the 8 settings gets 3-fold CV on the training split. The score is phishing F1. Dev is not used to pick the winner.


In [2]:
import pandas as pd

search.fit(data["X_train"], data["y_train"])

print("Best parameters:", search.best_params_)
print("Best CV phishing F1: {:.4f}".format(search.best_score_))

results = pd.DataFrame(search.cv_results_)
results = results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score")
results

Best parameters: {'model__C': 10.0, 'model__class_weight': None}
Best CV phishing F1: 0.8178


,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
6,10.0,None,0.817796,0.002666,1
4,1.0,None,0.817618,0.002652,2
2,0.1,None,0.817064,0.002635,3
0,0.01,None,0.815993,0.002617,4
7,10.0,balanced,0.780798,0.002280,5
5,1.0,balanced,0.780798,0.002240,6
3,0.1,balanced,0.780523,0.002197,7
1,0.01,balanced,0.773289,0.001448,8


Best CV result is C=10.0 with no class weight. Mean phishing F1 is 0.8178.

All four unweighted settings beat all four balanced ones. Balanced weights helped my recall in the investigation, but this search is going after F1, and the unweighted models do better there. C=10.0 is barely ahead of C=1.0 (0.8176). I have not scored dev yet.


## 3. Score it on dev

search.predict uses the refit winner, C=10.0 and no class weight. Dev still has the raw text columns. The pipeline drops those and scales the rest. Phishing is label 1. ROC AUC uses decision_function because LinearSVC does not have predict_proba.


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_dev = data["y_dev"]
dev_pred = search.predict(data["X_dev"])
dev_score = search.decision_function(data["X_dev"])

dev_metrics = {
    "accuracy": accuracy_score(y_dev, dev_pred),
    "precision": precision_score(y_dev, dev_pred, pos_label=1),
    "recall": recall_score(y_dev, dev_pred, pos_label=1),
    "f1": f1_score(y_dev, dev_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_dev, dev_score),
}

print("Best linear SVM dev scores")
print("Accuracy:           {:.4f}".format(dev_metrics["accuracy"]))
print("Phishing precision: {:.4f}".format(dev_metrics["precision"]))
print("Phishing recall:    {:.4f}".format(dev_metrics["recall"]))
print("Phishing F1:        {:.4f}".format(dev_metrics["f1"]))
print("ROC AUC:            {:.4f}".format(dev_metrics["roc_auc"]))

Best linear SVM dev scores
Accuracy:           0.9511
Phishing precision: 0.9170
Phishing recall:    0.7260
Phishing F1:        0.8104
ROC AUC:            0.9760


On dev the tuned SVM is about 95.11% accuracy, phishing F1 0.810, and ROC AUC 0.976. When it flags a URL it is right about 91.70% of the time, and it finds 72.60% of the phishing URLs.

Leaving class weights off is why precision is high and recall is lower. The search picked the best F1, not the model that catches the most phishing. Test is still unused.


## 4. Compared with my first SVM

My investigation SVM was C=0.01 and class_weight="balanced". Those dev numbers are from nipun_tree_svm_investigation.ipynb. The tuned model is C=10.0 with no class weight. Both rows are dev scores. Test is not in this table.


In [4]:
investigation_metrics = {
    "accuracy": 0.9226,
    "precision": 0.6669,
    "recall": 0.9244,
    "f1": 0.7748,
    "roc_auc": 0.9757,
}

comparison = pd.DataFrame(
    {
        "Investigation SVM": investigation_metrics,
        "Tuned SVM": dev_metrics,
    }
).T
comparison = comparison[["accuracy", "precision", "recall", "f1", "roc_auc"]]
comparison = comparison.rename(
    columns={
        "precision": "phishing_precision",
        "recall": "phishing_recall",
        "f1": "phishing_f1",
    }
)
comparison.round(4)

,accuracy,phishing_precision,phishing_recall,phishing_f1,roc_auc
Investigation SVM,0.9226,0.6669,0.9244,0.7748,0.9757
Tuned SVM,0.9511,0.9170,0.7260,0.8104,0.9760


## Conclusion

I would keep the tuned SVM. Phishing F1 goes from 0.775 on my first SVM to 0.810 here. Precision goes from 0.667 to 0.917, and accuracy from 92.26% to 95.11%. ROC AUC stays about 0.976 either way.

The downside is recall. The balanced SVM finds 92.44% of the phishing URLs. This one finds 72.60%. Turning the class weights off means fewer false alarms and more missed phishing sites.

For the group conclusion I am going with C=10.0 and no class weight, since we said we would pick by phishing F1. Test stays unused until conclusion.ipynb.
